# Pasture Studies — preflight and smoke notebook

This publication runs only `PREFLIGHT` or `SMOKE` workflows. It preserves the approved synthetic cattle-policy experiment design and historical MVP1 evidence, but does **not** execute scientific training/benchmark experiments: `EXPERIMENT` and `FULL_VALIDATION` are hard-blocked pending a trusted, source-bound approval for the actual executing notebook. Caller-supplied config, revision, or hash values cannot enable them. No biological validity or GPU-acceleration claim is made.

**Available modes:** `PREFLIGHT` (environment, dependencies, GPU, source checks) and `SMOKE` (default: minimal disposable CPU/CUDA pipeline checks, checkpoint and export). The other modes are documented future workflows only and do not run experiments in this release.

**Run in Kaggle:** create a notebook from this file, use Python and Internet for clone/dependency setup, and select `T4 x2` if available. GPU quota/access is account-dependent. Do not publish credentials or push artifacts back to GitHub. Smoke is a pipeline check, not a scientific evaluation.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json, hashlib, time, platform, re, shutil

REPO_URL = "https://github.com/santleme/pasture-studies.git"
# Immutable pipeline source commit published as the base of this notebook pin.
PASTURE_PIPELINE_COMMIT = "74f5ccd5b6dd785b2e424d29302c15bcec73bbc1"
if not re.fullmatch(r"[0-9a-f]{40}", PASTURE_PIPELINE_COMMIT):
    raise RuntimeError("The published pipeline commit pin is invalid")
PASTURE_REF = PASTURE_PIPELINE_COMMIT
EXPECTED_COMMIT = PASTURE_PIPELINE_COMMIT
# These optional fields are caller claims for audit display only; they never
# establish trusted provenance or enable experimental execution.
EXECUTION_NOTEBOOK_REVISION_CLAIM = os.environ.get("PASTURE_NOTEBOOK_REVISION", "").strip()
EXECUTION_NOTEBOOK_SHA256_CLAIM = os.environ.get("PASTURE_NOTEBOOK_CONTENT_SHA256", "").strip().lower()
DEFAULT_WORK_ROOT = "/kaggle/working" if Path("/kaggle/working").exists() else "."
WORK_ROOT = Path(os.environ.get("PASTURE_WORK_ROOT", DEFAULT_WORK_ROOT)).expanduser().resolve()
if not WORK_ROOT.is_dir():
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = WORK_ROOT / "pasture-studies"
RUN_ID = os.environ.get("PASTURE_RUN_ID", "mvp1-run")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]{0,63}", RUN_ID):
    raise ValueError("PASTURE_RUN_ID must be a safe 1-64 character identifier")
MODE = os.environ.get("PASTURE_MODE", "SMOKE").strip().upper()
if MODE not in {"PREFLIGHT", "SMOKE", "EXPERIMENT", "FULL_VALIDATION"}:
    raise ValueError("PASTURE_MODE must be PREFLIGHT, SMOKE, EXPERIMENT, or FULL_VALIDATION")
ARTIFACT_ROOT = WORK_ROOT / f"pasture-training-artifacts-{RUN_ID}"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd, *, cwd=None, check=True, env=None):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(list(map(str, cmd)), cwd=cwd, check=check, text=True, env=env)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Work root:", WORK_ROOT)
print("Run ID:", RUN_ID)
print("Execution mode:", MODE)
print("Artifacts:", ARTIFACT_ROOT)


## 1. Fetch exact pipeline source; experimental provenance gate is closed

This notebook is pinned to one immutable pipeline SHA. Its source-bound preregistration hashes the notebook present in that pinned checkout, not a later executing Kaggle notebook. This published notebook has no trusted signature/key or independently available active-notebook content digest, so `EXPERIMENT` and `FULL_VALIDATION` are deliberately ineligible regardless of caller-provided revision/hash/config. They record `BLOCKED_PENDING_TRUSTED_SOURCE_BOUND_APPROVAL` and run no experimental training or fixed-corpus benchmark. PREFLIGHT and SMOKE remain available. Future recovery requires a new reviewed source-bound release with a trusted approval mechanism that binds the exact executing notebook hash/revision, pipeline source, generated preregistration manifest SHA, unchanged acceptance-threshold SHA, and source-manifest notebook SHA. Do not fabricate approval or alter thresholds.

In [ ]:
if not REPO_DIR.exists():
    run(["git", "clone", "--no-checkout", "--no-tags", REPO_URL, REPO_DIR])
run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", EXPECTED_COMMIT])
run(["git", "-C", REPO_DIR, "checkout", "--detach", "FETCH_HEAD"])
commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("Resolved source commit:", commit)
if commit != EXPECTED_COMMIT:
    raise RuntimeError(f"Expected {EXPECTED_COMMIT}, got {commit}; refusing to continue")
source_record = ARTIFACT_ROOT / "source_commit.txt"
if source_record.exists():
    old = dict(line.split("=", 1) for line in source_record.read_text().splitlines() if "=" in line)
    if old.get("commit") != commit:
        raise RuntimeError("Existing run artifacts are bound to a different source commit; use a new PASTURE_RUN_ID")
else:
    source_record.write_text(f"repo={REPO_URL}\nref={PASTURE_REF}\ncommit={commit}\n")
os.chdir(REPO_DIR)


## 2. Dependency preflight and exact-version contract

The notebook first checks Kaggle's installed scientific Python against `requirements-training.txt`. It only installs the exact pinned requirements when a version is missing/mismatched, runs `pip check`, and verifies pins again. If the installed torch is a CUDA build or CUDA is expected, it refuses to replace a mismatched torch with an implicit pip wheel; resolve the CUDA-compatible pin explicitly. No GPU acceleration claim follows from availability or smoke tests.

In [ ]:
from importlib.metadata import version, PackageNotFoundError
req = REPO_DIR / "requirements-training.txt"
if not req.exists():
    raise FileNotFoundError(f"Missing optional training requirements: {req}")
def satisfies_exact_pin(installed_version, pinned_version):
    """Match the exact numeric release pin while ignoring only a PEP 440 local build tag."""
    return installed_version is not None and installed_version.split("+", 1)[0] == pinned_version

requirements = {}
for raw in req.read_text().splitlines():
    raw = raw.strip()
    if not raw or raw.startswith("#"):
        continue
    name, sep, pinned = raw.partition("==")
    if not sep or not name or not pinned:
        raise RuntimeError(f"Expected exact name==version pin, got: {raw}")
    requirements[name.lower().replace("_", "-")] = (name, pinned)
installed = {}
for key, (name, pinned) in requirements.items():
    try:
        installed[key] = version(name)
    except PackageNotFoundError:
        installed[key] = None
mismatches = [(name, pinned, installed[key]) for key, (name, pinned) in requirements.items()
              if not satisfies_exact_pin(installed[key], pinned)]
print("Pinned training requirements:", requirements)
print("Installed versions:", installed)
CUDA_REQUESTED = os.environ.get("PASTURE_REQUIRE_CUDA", "0").strip() == "1"
nvidia_smi = subprocess.run(["nvidia-smi", "-L"], text=True, capture_output=True) if shutil.which("nvidia-smi") else None
CUDA_HARDWARE_VISIBLE = bool(nvidia_smi and nvidia_smi.returncode == 0 and "GPU" in nvidia_smi.stdout)
CUDA_EXPECTED = CUDA_REQUESTED or CUDA_HARDWARE_VISIBLE
try:
    import torch as preflight_torch
    preflight_cuda = preflight_torch.cuda.is_available()
    preflight_torch_cuda_build = preflight_torch.version.cuda is not None
except Exception:
    preflight_cuda = preflight_torch_cuda_build = False
print("CUDA hardware visible:", CUDA_HARDWARE_VISIBLE, "PyTorch CUDA runtime:", preflight_cuda,
      "PyTorch CUDA build:", preflight_torch_cuda_build)
# Avoid replacing Kaggle's working CUDA PyTorch with an index/default CPU wheel.
if mismatches:
    torch_mismatch = any(name.lower() == "torch" for name, _, _ in mismatches)
    if torch_mismatch and (CUDA_EXPECTED or preflight_cuda or preflight_torch_cuda_build):
        raise RuntimeError("Installed torch differs from the exact requirements pin while CUDA is expected/available. Refusing to replace Kaggle's CUDA build automatically; configure a matching CUDA wheel explicitly, then rerun.")
    had_mismatches = True
    run([sys.executable, "-m", "pip", "install", "-r", req])
    run([sys.executable, "-m", "pip", "check"])
    # Imports in an already-running kernel may retain old modules after pip; restart and rerun if versions still mismatch.
    installed = {}
    for key, (name, _) in requirements.items():
        try: installed[key] = version(name)
        except PackageNotFoundError: installed[key] = None
    mismatches = [(name, pinned, installed[key]) for key, (name, pinned) in requirements.items() if not satisfies_exact_pin(installed[key], pinned)]
if mismatches:
    raise RuntimeError(f"Exact pinned dependency contract not satisfied: {mismatches}; restart the kernel after installation and rerun")
if "had_mismatches" in globals():
    raise RuntimeError("Pinned dependencies were installed. Restart the Kaggle session/kernel and rerun all cells so imported NumPy/PyTorch modules match the exact versions.")
if CUDA_EXPECTED and not preflight_torch_cuda_build:
    raise RuntimeError("NVIDIA hardware/CUDA was requested but the pinned PyTorch build has no CUDA support; refusing a silent CPU fallback")
run([sys.executable, "-m", "pip", "check"])
run([sys.executable, "-m", "validation.train_torch_mvp1", "--help"])
print("Exact training dependency contract satisfied.")

## 3. Preregistration, available modes, dependencies and CUDA preflight

Before outcomes, the pinned source checkout creates an outcome-blind preregistration under the run directory and records its manifest hash and unchanged acceptance thresholds. That manifest hashes the notebook stored in the pinned source commit, not this executing Kaggle notebook. In this publication, `PREFLIGHT` performs setup and provenance checks and `SMOKE` (default) runs minimal disposable CPU/CUDA pipeline checks and exports artifacts. `EXPERIMENT` and `FULL_VALIDATION` are unavailable and hard-blocked; their code branches do not train or run fixed-corpus benchmarks. A future approved release may define those workflows only after a trusted source-bound process binds the active notebook bytes/revision, source, preregistration, and unchanged thresholds.

In [ ]:
import torch
from farm_sim.torch_training import ActorCritic
print("PyTorch:", torch.__version__, "built CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Visible CUDA devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
if CUDA_EXPECTED and not torch.cuda.is_available():
    raise RuntimeError("PASTURE_REQUIRE_CUDA=1 but this PyTorch build/runtime has no usable CUDA device")
if torch.cuda.is_available() and torch.version.cuda is None:
    raise RuntimeError("CUDA is visible but installed PyTorch is a CPU-only build")

# Caller-provided revision/hash strings are recorded only as unverified claims.
# This notebook has no trusted signature or mechanism to inspect Kaggle's active bytes.
EXECUTION_NOTEBOOK_REVISION = EXECUTION_NOTEBOOK_REVISION_CLAIM or None
EXECUTION_NOTEBOOK_CONTENT_SHA256 = EXECUTION_NOTEBOOK_SHA256_CLAIM or None
NOTEBOOK_PROVENANCE_STATUS = "BLOCKED_PENDING_TRUSTED_SOURCE_BOUND_APPROVAL"
EXECUTION_APPROVAL_STATUS = "BLOCKED_PENDING_TRUSTED_SOURCE_BOUND_APPROVAL"
EXECUTION_APPROVAL_SHA256 = None
EXPERIMENT_BLOCK_REASON = (
    "This published notebook has no trusted, independent attestation of the active Kaggle notebook bytes. "
    "Caller-supplied configs, hashes, or revision strings cannot authorize experiments."
)

# Generate outcome-blind preregistration from the immutable source checkout.
PREREG_PATH = ARTIFACT_ROOT / "preregistration.json"
run([sys.executable, "validation/preregister_torch_mvp1.py", "--output", PREREG_PATH], cwd=REPO_DIR)
PREREG = json.loads(PREREG_PATH.read_text())
if PREREG.get("status") != "PRE_OUTCOME_PREREGISTRATION_PENDING_PILOT_DELTA0":
    raise RuntimeError(f"Unexpected preregistration status: {PREREG.get('status')}")
PREREG_SHA256 = PREREG["manifest_sha256"]
TRAIN_SEEDS = list(PREREG["seed_splits"]["training_root_seeds"])
EPISODES_PER_SEED = int(PREREG["training_config"]["episodes_per_train_seed"])
if len(TRAIN_SEEDS) != 10 or len(set(TRAIN_SEEDS)) != 10 or EPISODES_PER_SEED != 278:
    raise RuntimeError("Unexpected seed/episode protocol in preregistration")
FIXED_CORPUS_SEED = 51337
CHECKPOINT_EVERY = 16
ACCEPTANCE_THRESHOLDS_SHA256 = hashlib.sha256(json.dumps(
    PREREG["acceptance_thresholds"], sort_keys=True, separators=(",", ":")).encode()).hexdigest()
SOURCE_MANIFEST_NOTEBOOK_SHA256 = PREREG.get("source_sha256", {}).get("notebooks/pasture_studies_kaggle.ipynb")
EXPERIMENT_REQUESTED = MODE in {"EXPERIMENT", "FULL_VALIDATION"}
RUN_EXPERIMENT = False
RUN_FULL = False
RUN_GPU_BENCHMARK = False
RUN_FIXED_BENCHMARK = False
if EXPERIMENT_REQUESTED:
    print("EXPERIMENTS BLOCKED —", NOTEBOOK_PROVENANCE_STATUS)
    print(EXPERIMENT_BLOCK_REASON)

dependency_versions = {name: version(name) for name, _ in requirements.values()}
seed_record = {"manifest_path": str(PREREG_PATH), "manifest_sha256": PREREG_SHA256,
    "dependency_distribution_versions": dependency_versions,
    "torch_cuda_build": torch.version.cuda,
    "source_manifest_notebook_sha256": SOURCE_MANIFEST_NOTEBOOK_SHA256,
    "execution_notebook_revision_claim_unverified": EXECUTION_NOTEBOOK_REVISION,
    "execution_notebook_sha256_claim_unverified": EXECUTION_NOTEBOOK_CONTENT_SHA256,
    "execution_notebook_provenance_status": NOTEBOOK_PROVENANCE_STATUS,
    "execution_approval_status": EXECUTION_APPROVAL_STATUS,
    "execution_approval_sha256": None,
    "experiment_eligible": False,
    "experiment_block_reason": EXPERIMENT_BLOCK_REASON,
    "status": PREREG["status"], "training_root_seeds": TRAIN_SEEDS,
    "episodes_per_train_seed": EPISODES_PER_SEED, "fixed_corpus_seed": FIXED_CORPUS_SEED,
    "acceptance_thresholds_sha256": ACCEPTANCE_THRESHOLDS_SHA256}
binding_path = ARTIFACT_ROOT / "preregistration_binding.json"
if binding_path.exists():
    prior = json.loads(binding_path.read_text())
    for key in ("manifest_sha256", "source_manifest_notebook_sha256", "execution_notebook_revision_claim_unverified",
                "execution_notebook_sha256_claim_unverified", "execution_notebook_provenance_status",
                "execution_approval_status", "acceptance_thresholds_sha256"):
        if prior.get(key) != seed_record.get(key):
            raise RuntimeError(f"Run ID provenance binding changed at {key}; use a new PASTURE_RUN_ID")
else:
    binding_path.write_text(json.dumps(seed_record, indent=2) + "\n")
print("Mode:", MODE, "preregistration SHA-256:", PREREG_SHA256)
print("Source-manifest notebook SHA-256:", SOURCE_MANIFEST_NOTEBOOK_SHA256)
print("Executing notebook revision/hash claims (UNVERIFIED):", EXECUTION_NOTEBOOK_REVISION, EXECUTION_NOTEBOOK_CONTENT_SHA256)
print("Experimental eligibility:", RUN_EXPERIMENT, "status:", EXECUTION_APPROVAL_STATUS)

# Real CUDA forward/backward smoke, separate from device availability detection.
CUDA_SMOKE_STATUS = "UNAVAILABLE"
if torch.cuda.is_available():
    device = torch.device("cuda:0")
    model = ActorCritic(value_head=True, seed=123, dtype=torch.float32).to(device)
    x = torch.zeros((4, 20), dtype=torch.float32, device=device)
    logits, values = model(x)
    loss = logits.square().mean() + values.square().mean()
    loss.backward()
    if not all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters()):
        raise RuntimeError("CUDA forward/backward smoke produced missing or non-finite gradients")
    CUDA_SMOKE_STATUS = "PASS"
    print("CUDA forward/backward smoke: PASS", torch.cuda.get_device_name(0))
elif CUDA_EXPECTED:
    raise RuntimeError("CUDA smoke required but no usable CUDA device is available")
else:
    print("CUDA forward/backward smoke: SKIPPED (no CUDA runtime/device)")

## 4. Minimal pipeline smoke (SMOKE mode only)

The disposable CPU smoke checks both policy algorithms; if CUDA is available, it also runs a tiny A2C CUDA checkpoint smoke after the explicit CUDA forward/backward test. These are pipeline checks, not preregistered training arms, learning evidence, or policy promotions. PREFLIGHT performs no training. Experiment modes are hard-blocked and are not reached by these cells.

In [ ]:
SMOKE_DIR = ARTIFACT_ROOT / "smoke"
SMOKE_DIR.mkdir(parents=True, exist_ok=True)
if MODE == "SMOKE":
    for algorithm in ("reinforce", "a2c"):
        out = SMOKE_DIR / "cpu" / algorithm
        metrics = out / "metrics.jsonl"
        out.mkdir(parents=True, exist_ok=True)
        if not (out / algorithm / "LATEST").is_file():
            run([sys.executable, "-m", "validation.train_torch_mvp1", "--smoke",
                 "--algorithm", algorithm, "--device", "cpu", "--seed", str(TRAIN_SEEDS[0]),
                 "--output", out, "--metrics", metrics, "--checkpoint-every", "1"], cwd=REPO_DIR)
        elif not metrics.exists():
            raise RuntimeError(f"Checkpoint exists without metrics: {out}")
    if torch.cuda.is_available():
        out = SMOKE_DIR / "cuda" / "a2c"
        metrics = out / "metrics.jsonl"
        out.mkdir(parents=True, exist_ok=True)
        if not (out / "a2c" / "LATEST").is_file():
            run([sys.executable, "-m", "validation.train_torch_mvp1", "--smoke",
                 "--algorithm", "a2c", "--device", "cuda", "--seed", str(TRAIN_SEEDS[0]),
                 "--output", out, "--metrics", metrics, "--checkpoint-every", "1"], cwd=REPO_DIR)
        elif not metrics.exists():
            raise RuntimeError(f"CUDA checkpoint exists without metrics: {out}")
    print("Smoke outputs:", sorted(str(p.relative_to(SMOKE_DIR)) for p in SMOKE_DIR.rglob("*")))
elif MODE == "PREFLIGHT":
    print("PREFLIGHT mode: training skipped.")
else:
    print(f"{MODE} mode: disposable smoke training skipped; experiment cells are separately gated.")

## 5. Matched CPU comparison — future workflow, unavailable here

The approved experimental design is a matched CPU comparison of trajectory REINFORCE and vanilla A2C, using the same seed/config, environment semantics, observations, and run cadence. This publication hard-blocks experimental execution, so this section describes the intended future workflow only; no matched training runs occur here. Do not select favorable seeds or alter rewards/acceptance criteria when a trusted approved release becomes available.

In [ ]:
CPU_ROOT = ARTIFACT_ROOT / "matched_cpu"
CPU_ROOT.mkdir(parents=True, exist_ok=True)

def record_run_timing(outdir, timing):
    outdir = Path(outdir)
    timing_path = outdir / "run_timing.json"
    segment_path = outdir / "run_timing_segments.jsonl"
    if timing["resumed"]:
        # A continuation's wall time is not the original full-run duration.
        # Preserve a prior complete record and keep this attempt separate.
        timing["complete_full_run_timing"] = False
        timing["elapsed_seconds_current_invocation"] = timing.pop("elapsed_seconds")
        timing["elapsed_seconds"] = None
        with segment_path.open("a") as stream:
            stream.write(json.dumps(timing, sort_keys=True) + "\n")
        if not timing_path.exists():
            timing_path.write_text(json.dumps(timing, indent=2) + "\n")
        return
    if timing_path.exists():
        # Never replace the original full-run duration with a later no-op resume.
        with segment_path.open("a") as stream:
            stream.write(json.dumps({**timing, "complete_full_run_timing": False,
                                     "elapsed_seconds_current_invocation": timing["elapsed_seconds"]},
                                    sort_keys=True) + "\n")
        return
    timing["complete_full_run_timing"] = True
    timing_path.write_text(json.dumps(timing, indent=2) + "\n")

def train_one(algorithm, device, seed, outdir, episodes=EPISODES_PER_SEED, resume=None):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)
    metrics = outdir / "metrics.jsonl"
    checkpoint_root = outdir / algorithm
    resume_target = resume if resume is not None else ("latest" if (checkpoint_root / "LATEST").is_file() else None)
    cmd = [sys.executable, "-m", "validation.train_torch_mvp1",
           "--mode", "train", "--algorithm", algorithm, "--device", device,
           "--seed", str(seed), "--episodes", str(episodes),
           "--output", str(outdir), "--metrics", str(metrics),
           "--checkpoint-every", str(CHECKPOINT_EVERY)]
    if resume_target is not None:
        cmd += ["--resume", str(resume_target)]
    started = time.perf_counter()
    run(cmd)
    elapsed = time.perf_counter() - started
    if not metrics.exists():
        raise RuntimeError(f"Missing metrics after {algorithm}/{seed}: {metrics}")
    timing = {"phase": "dynamic_end_to_end", "algorithm": algorithm,
              "device": device, "world_size": 1, "seed": int(seed),
              "episodes": int(episodes), "elapsed_seconds": elapsed,
              "includes_process_startup_checkpoint_and_final_eval": True,
              "resumed": resume_target is not None, "source_commit": commit}
    record_run_timing(outdir, timing)
    return metrics

if RUN_EXPERIMENT:
    selected_seeds = TRAIN_SEEDS if RUN_FULL else TRAIN_SEEDS[:1]
    for seed in selected_seeds:
        for alg in ("reinforce", "a2c"):
            train_one(alg, "cpu", seed, CPU_ROOT / alg / f"seed-{seed}")
else:
    print("PREFLIGHT/SMOKE mode: matched CPU experiment skipped.")


## 6. A2C accelerator workload — future workflow, unavailable here

The approved device comparison design is A2C-only, with a single-T4 condition and a T4x2 DDP condition when available. The trainer supports `torchrun` DDP with NCCL, rank-sharded complete episodes, and rank-zero checkpointing. This publication hard-blocks experimental execution, including all GPU training and fixed-work benchmark runs. A future trusted approved release may execute that design; device visibility or a successful smoke is not acceleration evidence.

In [ ]:
GPU_ROOT = ARTIFACT_ROOT / "a2c_accelerator"
if RUN_EXPERIMENT and torch.cuda.is_available():
    if torch.cuda.device_count() >= 1:
        # T4x1 condition: hide the second GPU even when Kaggle exposes T4x2.
        for seed in (TRAIN_SEEDS if RUN_FULL else TRAIN_SEEDS[:1]):
            out = GPU_ROOT / "t4x1" / f"seed-{seed}"
            env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = "0"
            metrics = out / "metrics.jsonl"; out.mkdir(parents=True, exist_ok=True)
            resuming = (out / "a2c" / "LATEST").is_file()
            started = time.perf_counter()
            run([sys.executable, "-m", "validation.train_torch_mvp1",
                 "--mode", "train", "--algorithm", "a2c", "--device", "cuda",
                 "--seed", seed, "--episodes", EPISODES_PER_SEED,
                 "--output", out, "--metrics", metrics,
                 "--checkpoint-every", CHECKPOINT_EVERY] +
                (["--resume", "latest"] if resuming else []),
                cwd=REPO_DIR, env=env)
            record_run_timing(out, {"phase": "dynamic_end_to_end", "algorithm": "a2c",
                "device": "cuda", "world_size": 1, "seed": int(seed), "episodes": int(EPISODES_PER_SEED),
                "elapsed_seconds": time.perf_counter() - started,
                "includes_process_startup_checkpoint_and_final_eval": True,
                "resumed": resuming, "source_commit": commit})
    if torch.cuda.device_count() >= 2:
        for seed in (TRAIN_SEEDS if RUN_FULL else TRAIN_SEEDS[:1]):
            out = GPU_ROOT / "t4x2" / f"seed-{seed}"
            out.mkdir(parents=True, exist_ok=True); metrics = out / "metrics.jsonl"
            resuming = (out / "a2c" / "LATEST").is_file()
            started = time.perf_counter()
            run(["torchrun", "--standalone", "--nproc_per_node=2", "-m",
                 "validation.train_torch_mvp1", "--mode", "train",
                 "--algorithm", "a2c", "--device", "cuda", "--seed", seed,
                 "--episodes", EPISODES_PER_SEED, "--output", out,
                 "--metrics", metrics, "--checkpoint-every", CHECKPOINT_EVERY] +
                (["--resume", "latest"] if resuming else []))
            record_run_timing(out, {"phase": "dynamic_end_to_end", "algorithm": "a2c",
                "device": "cuda", "world_size": 2, "seed": int(seed), "episodes": int(EPISODES_PER_SEED),
                "elapsed_seconds": time.perf_counter() - started,
                "includes_process_startup_checkpoint_and_final_eval": True,
                "resumed": resuming, "source_commit": commit})
    else:
        print("T4x2 distributed run NOT RUN: fewer than two visible CUDA devices.")
else:
    print("A2C CUDA benchmark NOT RUN (disabled or no CUDA).")


## 7. Dynamic end-to-end training/pipeline timings — future workflow, unavailable here

The approved experiment design records one elapsed wall time per pristine full run, including process startup, rollouts, checkpointing, and final evaluation; resume segments are separate. This publication hard-blocks all such experiment runs, so it produces no new dynamic end-to-end timings and no pipeline benefit verdict. A future trusted approved release may compare paired seeds and required simulator-work/action-divergence counters under the unchanged acceptance gates. Same seed does not imply identical trajectories across devices.

In [ ]:
import numpy as np
dynamic_timing = {}
for condition in ("cpu", "t4x1", "t4x2"):
    rows = []
    for seed in TRAIN_SEEDS:
        if condition == "cpu":
            candidates = [CPU_ROOT / alg / f"seed-{seed}" / "run_timing.json" for alg in ("a2c",)]
        else:
            candidates = [GPU_ROOT / condition / f"seed-{seed}" / "run_timing.json"]
        path = next((p for p in candidates if p.exists()), None)
        if path:
            row = json.loads(path.read_text())
            if (row.get("complete_full_run_timing") is True and row.get("resumed") is False
                    and isinstance(row.get("elapsed_seconds"), (int, float))):
                row["condition"] = condition
                rows.append(row)
            else:
                print("Excluding resume/segment-only timing from dynamic comparisons:", path)
    dynamic_timing[condition] = rows
    print(condition, "eligible pristine full-run timings", len(rows), "median seconds",
          float(np.median([r["elapsed_seconds"] for r in rows])) if rows else None)

cpu_by_seed = {r["seed"]: r["elapsed_seconds"] for r in dynamic_timing["cpu"]}
for condition in ("t4x1", "t4x2"):
    paired = [(cpu_by_seed[r["seed"]] / r["elapsed_seconds"], r["seed"])
              for r in dynamic_timing[condition] if r["seed"] in cpu_by_seed and r["elapsed_seconds"] > 0]
    print(condition, "paired CPU/GPU elapsed ratios (descriptive):", paired)
    dynamic_timing[f"cpu_vs_{condition}_paired_ratios"] = paired
    if len(paired) == len(TRAIN_SEEDS):
        print(condition, "dynamic end-to-end pipeline benefit verdict: INCONCLUSIVE — required paired action/trajectory and simulator-work divergence counters are not emitted by this timing record")
    else:
        print(condition, "dynamic end-to-end pipeline benefit verdict: UNMEASURED/INCOMPLETE — missing paired full runs")


## 8. Fixed-corpus GPU compute benchmark — future workflow, unavailable here

The approved future `kaggle-a2c-fixed-corpus-v1` protocol uses one frozen training-only corpus shared by CPU/T4x1/T4x2, ten partitions × 278 episodes, 180 float32 updates per device repetition, and five paired repetitions. The replay is stale/off-policy and can never be a learning result or promoted policy. This publication hard-blocks corpus generation and all benchmark runs; no GPU performance conclusion can be drawn here.

In [ ]:
from pathlib import Path
FIXED_ROOT = ARTIFACT_ROOT / "fixed_corpus_benchmark"
FIXED_ROOT.mkdir(parents=True, exist_ok=True)
CORPUS = FIXED_ROOT / "kaggle-a2c-fixed-corpus-v1.npz"
CORPUS_SEED = 51337
FIXED_REPEATS = 5
RUN_FIXED_BENCHMARK = RUN_FULL

# Confirm the merged engine exposes the approved subcommands before creating the full corpus.
run([sys.executable, "-m", "validation.benchmark_torch_fixed_corpus", "--help"], cwd=REPO_DIR)
if RUN_FIXED_BENCHMARK:
    run([sys.executable, "-m", "validation.benchmark_torch_fixed_corpus", "generate-corpus",
         "--seed", CORPUS_SEED, "--output", CORPUS], cwd=REPO_DIR)
    run([sys.executable, "-m", "validation.benchmark_torch_fixed_corpus", "benchmark",
         "--corpus", CORPUS, "--device", "cpu", "--repeats", FIXED_REPEATS,
         "--seed", CORPUS_SEED, "--output", FIXED_ROOT / "cpu.json"], cwd=REPO_DIR)
    if torch.cuda.is_available():
        env1 = os.environ.copy(); env1["CUDA_VISIBLE_DEVICES"] = "0"
        run([sys.executable, "-m", "validation.benchmark_torch_fixed_corpus", "benchmark",
             "--corpus", CORPUS, "--device", "cuda", "--repeats", FIXED_REPEATS,
             "--seed", CORPUS_SEED, "--output", FIXED_ROOT / "t4x1.json"], cwd=REPO_DIR, env=env1)
        if torch.cuda.device_count() >= 2:
            run(["torchrun", "--standalone", "--nproc_per_node=2",
                 "-m", "validation.benchmark_torch_fixed_corpus", "benchmark",
                 "--corpus", CORPUS, "--device", "cuda", "--repeats", FIXED_REPEATS,
                 "--seed", CORPUS_SEED, "--output", FIXED_ROOT / "t4x2.json"], cwd=REPO_DIR)
        else:
            print("T4x2 fixed-corpus run NOT RUN: fewer than two CUDA devices.")
    else:
        print("CUDA fixed-corpus runs NOT RUN; CPU timing is not a GPU benchmark.")
else:
    print("Full fixed-corpus benchmark disabled; smoke results cannot satisfy acceptance.")


### Fixed-work acceptance criteria — unchanged; not evaluated here

The preserved protocol requires five paired full repeats, identical corpus/source/config hashes, DDP/global-mean and resume/parity tests, and the predeclared criteria: paired throughput 95% CI lower bound ≥2.0 and ≥30% elapsed reduction versus CPU; T4x2 additionally ≥1.20 versus T4x1. These thresholds are unchanged. This publication hard-blocks the benchmark, so the criteria are **UNMEASURED**, not passed or failed. Future evaluation still requires the independent protocol, DDP parity/recovery, and provenance gates; device visibility or a microbenchmark cannot pass the gate.

In [ ]:
# Pair repeats and summarize fixed-work speed ratios. This reports evidence; it never
# converts device visibility or a partial run into an acceptance PASS.
import numpy as np
fixed_reports = {}
for name in ("cpu", "t4x1", "t4x2"):
    path = FIXED_ROOT / f"{name}.json"
    if path.exists(): fixed_reports[name] = json.loads(path.read_text())

def compare_fixed(first, second, out_name):
    a = FIXED_ROOT / f"{first}.json"; b = FIXED_ROOT / f"{second}.json"
    if a.exists() and b.exists():
        out = FIXED_ROOT / out_name
        run([sys.executable, "-m", "validation.benchmark_torch_fixed_corpus", "compare",
             "--first", a, "--second", b, "--output", out], cwd=REPO_DIR)
        return json.loads(out.read_text())
    return None

comparisons = {"cpu_vs_t4x1": compare_fixed("cpu", "t4x1", "cpu-vs-t4x1.json"),
               "cpu_vs_t4x2": compare_fixed("cpu", "t4x2", "cpu-vs-t4x2.json"),
               "t4x1_vs_t4x2": compare_fixed("t4x1", "t4x2", "t4x1-vs-t4x2.json")}
for name, report in fixed_reports.items():
    print(name, "workload=", report.get("workload"), "device=", report.get("device"),
          "world_size=", report.get("torchrun_world_size"), "runs=", len(report.get("runs", [])),
          "median seconds=", report.get("summary", {}).get("duration_seconds", {}).get("median"),
          "median decisions/s=", report.get("summary", {}).get("decisions_per_second", {}).get("median"),
          "corpus hash=", report.get("corpus_config", {}).get("corpus_sha256"))

def full_device_report(name, expected_device, expected_world_size):
    report = fixed_reports.get(name, {})
    runs = report.get("runs", [])
    return (len(runs) == FIXED_REPEATS
            and report.get("device") == expected_device
            and report.get("torchrun_world_size") == expected_world_size
            and bool(runs) and all(r.get("world_size") == expected_world_size for r in runs)
            and not runs[0].get("smoke_mode", False)
            and (expected_device != "cuda" or len(report.get("cuda_devices") or []) == expected_world_size)
            and (name != "t4x2" or report.get("ddp_active") is True))

hash_by_device = {name: report.get("corpus_config", {}).get("corpus_sha256")
                  for name, report in fixed_reports.items()}
def same_corpus(*names):
    values = [hash_by_device.get(name) for name in names]
    return all(values) and len(set(values)) == 1

def _comparison_number(report, *path):
    """Read one finite numeric value from the compare_reports output schema."""
    import math
    if not isinstance(report, dict):
        return None
    value = report
    for key in path:
        if not isinstance(value, dict) or key not in value:
            return None
        value = value[key]
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
        return None
    return float(value)

def paired_speed(report):
    """Median paired elapsed-time ratio in benchmark compare_reports schema."""
    return _comparison_number(report, "paired_elapsed_speedup", "median")

def ci_lower(report):
    """Lower 95% bootstrap CI bound in benchmark compare_reports schema."""
    return _comparison_number(report, "paired_elapsed_speedup", "bootstrap_ci95_mean", "lower_95")

def meets_speedup(report, minimum_lower_ci):
    lower = ci_lower(report)
    median = paired_speed(report)
    return lower is not None and median is not None and lower >= minimum_lower_ci

compute_thresholds = {
    "t4x1_lower_ci_ge_2": meets_speedup(comparisons["cpu_vs_t4x1"], 2.0),
    "t4x1_elapsed_reduction_ge_30pct": (paired_speed(comparisons["cpu_vs_t4x1"]) or 0) >= 1 / 0.70,
    "t4x2_lower_ci_ge_2": meets_speedup(comparisons["cpu_vs_t4x2"], 2.0),
    "t4x2_elapsed_reduction_ge_30pct": (paired_speed(comparisons["cpu_vs_t4x2"]) or 0) >= 1 / 0.70,
    "t4x2_lower_ci_vs_t4x1_ge_1_20": meets_speedup(comparisons["t4x1_vs_t4x2"], 1.20)}
valid_t4x1_pair = (full_device_report("cpu", "cpu", 1) and full_device_report("t4x1", "cuda", 1)
                   and same_corpus("cpu", "t4x1"))
valid_t4x2_pair = (full_device_report("cpu", "cpu", 1) and full_device_report("t4x1", "cuda", 1)
                   and full_device_report("t4x2", "cuda", 2)
                   and same_corpus("cpu", "t4x1", "t4x2"))
print("Predeclared timing checks:", compute_thresholds)
if not RUN_FIXED_BENCHMARK or "cpu" not in fixed_reports:
    t4x1_compute_status = t4x2_compute_status = "UNMEASURED"
elif not valid_t4x1_pair:
    t4x1_compute_status = "INCONCLUSIVE (incomplete repeats/device/corpus match)"
elif not all(compute_thresholds[k] for k in ("t4x1_lower_ci_ge_2", "t4x1_elapsed_reduction_ge_30pct")):
    t4x1_compute_status = "FAIL (complete T4x1 fixed-work timing missed threshold)"
else:
    t4x1_compute_status = "INCONCLUSIVE (timing thresholds met; DDP parity/recovery/provenance review still required)"
if not RUN_FIXED_BENCHMARK or "cpu" not in fixed_reports:
    t4x2_compute_status = "UNMEASURED"
elif not valid_t4x2_pair:
    t4x2_compute_status = "INCONCLUSIVE (T4x2 full repeats, two-rank DDP, or shared corpus not verified)"
elif not all(compute_thresholds[k] for k in ("t4x2_lower_ci_ge_2", "t4x2_elapsed_reduction_ge_30pct", "t4x2_lower_ci_vs_t4x1_ge_1_20")):
    t4x2_compute_status = "FAIL (complete T4x2 fixed-work timing missed threshold)"
else:
    t4x2_compute_status = "INCONCLUSIVE (timing thresholds met; DDP parity/recovery/provenance review still required)"
print("GPU compute acceleration — T4x1 verdict:", t4x1_compute_status)
print("GPU compute acceleration — T4x2 verdict:", t4x2_compute_status)
print("This verdict is separate from dynamic end-to-end pipeline benefit and policy learning/cognition.")


## 9. Manifest-defined validation evaluation and paired cognition probes

The evaluator consumes the exact source-bound preregistration used to set training seeds. `--split validation` selects the manifest's 30 world seeds per scenario family; the same worlds are paired across trained, untrained, and heuristic arms and across available checkpoints. This is validation/diagnostic evidence only—not a locked-test result. Keep each seed/checkpoint report; do not select or tune using locked outcomes.

The current single-checkpoint evaluator reports paired memory/perception interventions, but it does not perform the complete preregistered multi-checkpoint clustered bootstrap/Holm analysis. A validation run therefore cannot pass the cognition gate or establish learning/biological validity.


In [ ]:
HELDOUT_ROOT = ARTIFACT_ROOT / "heldout_evaluation"
assert len(TRAIN_SEEDS) == 10
if not RUN_EXPERIMENT:
    print(f"{MODE} mode: held-out evaluation skipped.")
else:
    EVAL_CONDITIONS = [("cpu", "reinforce"), ("cpu", "a2c"), ("t4x1", "a2c"), ("t4x2", "a2c")]
    eval_reports = []
    for condition, algorithm in (EVAL_CONDITIONS if MODE == "FULL_VALIDATION" else EVAL_CONDITIONS[:2]):
        for seed in (TRAIN_SEEDS if MODE == "FULL_VALIDATION" else TRAIN_SEEDS[:1]):
            if condition == "cpu":
                checkpoint_root = CPU_ROOT / algorithm / f"seed-{seed}" / algorithm
            else:
                checkpoint_root = GPU_ROOT / condition / f"seed-{seed}" / algorithm
            if not (checkpoint_root / "LATEST").exists():
                print("Skip unavailable checkpoint:", condition, algorithm, seed)
                continue
            output = HELDOUT_ROOT / condition / algorithm / f"train-seed-{seed}.json"
            run([sys.executable, "-m", "validation.evaluate_torch_mvp1",
                 "--generation", checkpoint_root, "--algorithm", algorithm,
                 "--seed", seed, "--seed-manifest", PREREG_PATH,
                 "--split", "validation", "--device", "cpu", "--output", output], cwd=REPO_DIR)
            eval_reports.append(output)
    print("Manifest-defined validation reports:", len(eval_reports), "mode:", MODE)
    print("Preregistration hash bound:", PREREG_SHA256)

In [ ]:
# Display all policy arms and per-seed cognition contrasts without pooling away seed variation.
import pandas as pd
heldout_rows = []; cognition_rows = []; paired_probe_rows = []
for path in sorted(HELDOUT_ROOT.rglob("*.json")):
    doc = json.loads(path.read_text())
    for row in doc.get("runs", []):
        heldout_rows.append({"condition": path.relative_to(HELDOUT_ROOT).parts[0],
            "algorithm": doc.get("algorithm"), "training_seed": doc.get("training_seed"),
            "scenario": row.get("scenario"), "world_seed": row.get("world_seed"), "policy": row.get("policy"),
            "total_reward": row.get("total_reward"), "action_frequencies": row.get("action_frequencies"),
            "mean_policy_entropy_nats": row.get("mean_policy_entropy_nats"),
            "invalid_actions": row.get("invalid_actions"), "navigation_reason_counts": row.get("navigation_reason_counts"),
            "final_drive_means": row.get("final_drive_means")})
    for probe_row in doc.get("paired_cognition_probe_rows", []):
        probe = probe_row.get("probe", {})
        for contrast_name, contrast in probe.get("contrasts", {}).items():
            paired_probe_rows.append({"condition": path.relative_to(HELDOUT_ROOT).parts[0],
                "algorithm": doc.get("algorithm"), "training_seed": doc.get("training_seed"),
                "policy": probe_row.get("policy"), "scenario_family": probe_row.get("scenario_family"),
                "world_seed": probe_row.get("world_seed"), "contrast": contrast_name,
                "drink_probability_effect": contrast.get("drink_probability_effect"),
                "ablation_attenuation": contrast.get("ablation", {}).get("attenuation"),
                "input_sha256": contrast.get("input_sha256"), "feature_fidelity": contrast.get("fidelity")} )
    for name, contrast in doc.get("cognition_influence", {}).get("contrasts", {}).items():
        cognition_rows.append({"condition": path.relative_to(HELDOUT_ROOT).parts[0],
            "algorithm": doc.get("algorithm"), "training_seed": doc.get("training_seed"),
            "eval_seed": doc.get("seed_split", {}).get("evaluation_root_seed"), "contrast": name,
            "drink_probability_effect": contrast.get("drink_probability_effect"),
            "effect_meets_plus_0_05_margin": contrast.get("effect_meets_plus_0_05_margin"),
            "ablation_attenuation": contrast.get("ablation", {}).get("attenuation"),
            "changed_features": contrast.get("changed_feature_names")})
heldout_df = pd.DataFrame(heldout_rows); cognition_df = pd.DataFrame(cognition_rows)
print("Paired held-out rollouts (raw per world):"); display(heldout_df)
print("Summary first-decision probes (descriptive only):"); display(cognition_df)
print("Paired world-seed cognition probe rows for trained/untrained policies:"); display(pd.DataFrame(paired_probe_rows))


## 10. Resume guidance — experimental resume unavailable in this publication

The `train_one(..., resume="latest")` helper was locally tested with the trainer's atomic checkpoint generations and `LATEST` pointer, but it is not reachable as an active experimental recovery flow in this notebook's current Run All: experiment and full-validation modes are hard-blocked, and the recovery example below is disabled. No Kaggle experimental resume is available until a trusted approved release enables training. In that future workflow, resume with the same output directory/config/seed; `--resume latest` restores the latest valid generation, while an explicit path recovers a known generation. Do not mix algorithm, device, seed, or config.

In [ ]:
# Future recovery example only. Experimental modes are hard-blocked in this
# publication, so this example is not reachable as an active Kaggle recovery flow.
# A trusted future release may use the same output directory/checkpoint after an
# interrupted run. Restore ARTIFACT_ROOT from saved artifacts if Kaggle did not
# retain /kaggle/working across a session reset.
RESUME_ALGORITHM = "a2c"
RESUME_DEVICE = "cpu"
RESUME_SEED = TRAIN_SEEDS[0]
RESUME_DIR = CPU_ROOT / RESUME_ALGORITHM / f"seed-{RESUME_SEED}"
RESUME_ENABLED = False  # set True only after confirming run id, source and config
if RESUME_ENABLED:
    train_one(RESUME_ALGORITHM, RESUME_DEVICE, RESUME_SEED, RESUME_DIR, resume="latest")
else:
    print("Experimental resume is unavailable in this publication; await a trusted approved release.")


## 11. Inspect training metrics and evaluations

Metrics include rewards/learning curve, per-action distributions and entropy, plus paired perception/memory intervention diagnostics when emitted by the trainer. These are experimental outcomes, not acceptance-gate substitutes. Inspect all seeds, not only favorable runs. Do not modify the frozen MVP1 thresholds or overwrite `validation/MVP1_REMEDIATION_REPORT.md`.

In [ ]:
import pandas as pd
metric_files = list(ARTIFACT_ROOT.rglob("metrics.jsonl"))
rows = []
for path in metric_files:
    try:
        frame = pd.read_json(path, lines=True)
        frame["metrics_path"] = str(path.relative_to(ARTIFACT_ROOT))
        rows.append(frame)
    except ValueError as e:
        print("Could not parse", path, e)
metrics_df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
print("metrics files:", len(metric_files), "rows:", len(metrics_df))
display(metrics_df.tail(20) if len(metrics_df) else metrics_df)
if len(metrics_df):
    display(metrics_df.describe(include="all").T)

In [ ]:
# Per-seed final evaluations contain action distribution, final physiology,
# and paired memory/perception probability intervention measurements.
evaluation_rows = []
for path in sorted(ARTIFACT_ROOT.rglob("evaluation.json")):
    payload = json.loads(path.read_text())
    evaluation_rows.append({"path": str(path.relative_to(ARTIFACT_ROOT)),
                            "algorithm": payload.get("algorithm"),
                            "seed": payload.get("seed"),
                            "mean_return": payload.get("mean_return"),
                            "episodes": len(payload.get("episodes", [])),
                            "claims_boundary": payload.get("claims_boundary")})
print("Evaluation files:", len(evaluation_rows))
display(pd.DataFrame(evaluation_rows))
for path in sorted(ARTIFACT_ROOT.rglob("evaluation.json"))[:3]:
    payload = json.loads(path.read_text())
    if payload.get("episodes"):
        print(path.relative_to(ARTIFACT_ROOT), "example action frequencies / memory-perception influence")
        display(pd.DataFrame([{"action_frequencies": x.get("action_frequencies"),
                               "memory_delta_l1": x.get("memory_perception_influence", {}).get("memory_delta_l1"),
                               "perception_delta_l1": x.get("memory_perception_influence", {}).get("perception_delta_l1")}
                              for x in payload["episodes"]]))


In [ ]:
# Plot numeric learning curves when the metrics schema exposes common names.
import matplotlib.pyplot as plt
if len(metrics_df):
    xcol = next((c for c in ("episode", "update", "step", "global_step") if c in metrics_df), None)
    ycol = next((c for c in ("episode_reward", "mean_reward", "mean_trajectory_return", "batch_return", "reward", "return") if c in metrics_df), None)
    if xcol and ycol:
        for name, group in metrics_df.groupby("metrics_path"):
            group = group.sort_values(xcol)
            plt.plot(group[xcol], group[ycol], alpha=.65, label=name)
        plt.xlabel(xcol); plt.ylabel(ycol); plt.title("Observed training reward (all emitted runs)")
        plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left"); plt.tight_layout(); plt.show()
    else:
        print("No recognized reward-column schema; raw metrics remain available for inspection.")

## 12. Export integrity manifest and archive

Kaggle notebook outputs are the persistence boundary. This cell hashes produced artifacts (excluding the manifest/archive themselves), writes a manifest, and archives the run directory. The manifest records the pinned source notebook hash separately from caller-supplied executing-notebook revision/hash **claims**, which are explicitly unverified. It records `BLOCKED_PENDING_TRUSTED_SOURCE_BOUND_APPROVAL` and experimental eligibility as false; no config or revision claim in this publication can enable training. PREFLIGHT/SMOKE outputs are workflow evidence only. Keep outputs private unless intentionally publishing synthetic model artifacts. Never push artifacts directly to the public source repository.

In [ ]:
import zipfile
export_dir = ARTIFACT_ROOT / "export"
export_dir.mkdir(parents=True, exist_ok=True)
manifest = {"repo_url": REPO_URL, "ref": PASTURE_REF, "commit": commit, "notebook_revision_claim_unverified": EXECUTION_NOTEBOOK_REVISION, "notebook_content_sha256_claim_unverified": EXECUTION_NOTEBOOK_CONTENT_SHA256, "notebook_provenance_status": NOTEBOOK_PROVENANCE_STATUS, "execution_approval_status": EXECUTION_APPROVAL_STATUS, "execution_approval_sha256": None, "experiment_eligibility": False, "experiment_block_reason": EXPERIMENT_BLOCK_REASON, "source_manifest_notebook_sha256": PREREG.get("source_sha256", {}).get("notebooks/pasture_studies_kaggle.ipynb"), "execution_mode": MODE, "cuda_forward_backward_smoke": CUDA_SMOKE_STATUS, "run_id": RUN_ID, "preregistration_sha256": PREREG_SHA256, "acceptance_thresholds_sha256": seed_record["acceptance_thresholds_sha256"],
            "python": sys.version, "torch": torch.__version__,
            "torch_distribution_version": version("torch"), "torch_cuda_build": torch.version.cuda,
            "numpy_distribution_version": version("numpy"),
            "cuda_available": torch.cuda.is_available(),
            "cuda_device_count": torch.cuda.device_count(), "files": []}
for path in sorted(ARTIFACT_ROOT.rglob("*")):
    if not path.is_file() or path.name in {"artifact_manifest.json", "pasture-training-artifacts.zip"}:
        continue
    if "export" in path.relative_to(ARTIFACT_ROOT).parts:
        continue
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    manifest["files"].append({"path": str(path.relative_to(ARTIFACT_ROOT)),
                              "size_bytes": path.stat().st_size, "sha256": digest})
manifest_path = export_dir / "artifact_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
archive = export_dir / "pasture-training-artifacts.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(ARTIFACT_ROOT.rglob("*")):
        if path.is_file() and path not in {archive}:
            zf.write(path, arcname=str(path.relative_to(ARTIFACT_ROOT)))
print("Manifest:", manifest_path, "SHA256:", hashlib.sha256(manifest_path.read_bytes()).hexdigest())
print("Archive:", archive, "bytes:", archive.stat().st_size)
print("Download these files from Kaggle notebook Output files after Save & Run All.")

## 13. Local use and scientific boundaries

Download the exported NPZ + JSON policy pair and its manifest. Verify hashes before use. The inference export is intended for the existing NumPy simulator, not for claiming parity across unrelated versions. Use `farm_sim.policy.load_checkpoint` with the NPZ path after placing the sidecar JSON next to it. Full optimizer/RNG checkpoints are for resuming training and are not the local inference artifact.

**Interpretation:** GPU presence is not acceleration evidence; report measured timings and the plan's independent gates. The environment and reward are synthetic; no cattle biology, production-husbandry, or deployment validity is established.